# Prétraitement de données textuelles

**Dataset** : Allocine — avis de films en français, étiquetés **Positif** ou **Négatif**.

| Étape | Contenu |
|---|---|
| 1 | Chargement et exploration |
| 2 | Prétraitement (nettoyage → tokenisation → stopwords → stemming / lemmatisation) |
| 3 | Représentation vectorielle (BoW, TF-IDF) |
| 4 | Classification (Naïf de Bayes) |
| 5 | Évaluation (Précision, Rappel, F1) |
| 6 | Bonus — comparaison des configurations |

---
## Installation des dépendances

In [ ]:
!pip install nltk scikit-learn spacy --quiet
!python -m spacy download fr_core_news_sm --quiet

---
## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('NLTK prêt.')

In [ ]:
import spacy
nlp = spacy.load('fr_core_news_sm')
print('SpaCy prêt — modèle :', nlp.meta['name'])

---
## Étape 1 — Chargement et exploration du dataset

In [ ]:
df = pd.read_csv('allocine_bruite.csv')
df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)
print('Train :', df_train.shape, '| Test :', df_test.shape)

In [ ]:
df_train.head(3)

In [ ]:
print(df_train['sentiment'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
counts = df_train['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#E74C3C', '#2ECC71'])
axes[0].set_title('Distribution des sentiments')
df_train['nb_mots'] = df_train['review'].apply(lambda x: len(x.split()))
for label, color in [('Négatif', '#E74C3C'), ('Positif', '#2ECC71')]:
    df_train[df_train['sentiment'] == label]['nb_mots'].plot(
        kind='kde', ax=axes[1], label=label, color=color)
axes[1].set_title('Longueur des avis (mots)')
axes[1].legend()
plt.tight_layout(); plt.show()

---
## Étape 2 — Prétraitement

| Sous-étape | Outil |
|---|---|
| Nettoyage regex | `re` |
| Tokenisation | NLTK `word_tokenize` / SpaCy |
| Stopwords | NLTK + SpaCy (union) |
| Stemming | NLTK `SnowballStemmer` |
| Lemmatisation | SpaCy `fr_core_news_sm` |

### 2.1 Nettoyage regex

In [ ]:
# Sélection d'un exemple bruitée
texte_brut = df_train['review'].iloc[0]
print(texte_brut[:300])

In [ ]:
def nettoyer(texte):
    texte = re.sub(r'<[^>]+>',   ' ', texte)   # balises HTML
    texte = re.sub(r'https?\S+', ' ', texte)   # URLs
    texte = re.sub(r'@\w+',      ' ', texte)   # mentions @user
    texte = re.sub(r'#\w+',      ' ', texte)   # hashtags
    texte = re.sub(r'\d+',       ' ', texte)   # chiffres
    texte = re.sub(r'[^\w\s]',   ' ', texte)   # ponctuation
    texte = re.sub(r'\s+',       ' ', texte)   # espaces multiples
    return texte.strip()

In [ ]:
texte_net = nettoyer(texte_brut)
print('BRUT    :', texte_brut[:200])
print()
print('NETTOYÉ :', texte_net[:200])

### 2.2 Tokenisation

Deux bibliothèques permettent de tokeniser en français : **NLTK** et **SpaCy**.

In [ ]:
from nltk.tokenize import word_tokenize

tokens_nltk = word_tokenize(texte_net, language='french')
print(f'NLTK — {len(tokens_nltk)} tokens : {tokens_nltk[:15]}')

In [ ]:
tokens_spacy = [t.text for t in nlp(texte_net)]
print(f'SpaCy — {len(tokens_spacy)} tokens : {tokens_spacy[:15]}')

In [ ]:
# On garde NLTK pour la suite : mots alpha, minuscules, longueur > 2
tokens_alpha = [t.lower() for t in tokens_nltk if t.isalpha() and len(t) > 2]
print(f'Après filtre alpha/minuscules : {len(tokens_alpha)} tokens')
print(tokens_alpha[:20])

### 2.3 Stopwords

NLTK et SpaCy ont chacun leur liste. On fait l'**union** pour une meilleure couverture.

In [ ]:
from nltk.corpus import stopwords

stop_nltk = set(stopwords.words('french'))
print(f'NLTK  : {len(stop_nltk)} stopwords — ex : {sorted(stop_nltk)[:8]}')

In [ ]:
stop_spacy = nlp.Defaults.stop_words
print(f'SpaCy : {len(stop_spacy)} stopwords — ex : {sorted(stop_spacy)[:8]}')

In [ ]:
stop_fr = stop_nltk | stop_spacy
print(f'Union NLTK ∪ SpaCy : {len(stop_fr)} stopwords')

In [ ]:
tokens_clean = [t for t in tokens_alpha if t not in stop_fr]
print(f'Avant : {len(tokens_alpha)} tokens → Après : {len(tokens_clean)} tokens')
print('Tokens nets :', tokens_clean[:20])

### 2.4 Stemming — NLTK `SnowballStemmer`

Troncature rapide vers la racine. Approximatif : la racine n'est pas toujours un vrai mot.

In [ ]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer('french')

In [ ]:
mots_demo = ['acteurs', 'jouaient', 'magnifique', 'connexion', 'catastrophique', 'réalisateur']
print(f"{'Mot original':<22} {'Racine (stem)'}")
print('-' * 38)
for m in mots_demo:
    print(f"{m:<22} {stemmer.stem(m)}")

In [ ]:
tokens_stem = [stemmer.stem(t) for t in tokens_clean]
print('Avant  :', tokens_clean[:12])
print('Stems  :', tokens_stem[:12])

### 2.5 Lemmatisation — SpaCy `fr_core_news_sm`

Analyse morphologique : réduit à la **forme canonique** (vrai mot du dictionnaire).

In [ ]:
doc_demo = nlp(' '.join(mots_demo))
print(f"{'Mot original':<22} {'Lemme':<22} {'POS (nature)'}")
print('-' * 56)
for tok in doc_demo:
    print(f"{tok.text:<22} {tok.lemma_:<22} {tok.pos_}")

In [ ]:
doc_clean = nlp(' '.join(tokens_clean))
tokens_lemme = [tok.lemma_.lower() for tok in doc_clean
                if not tok.is_stop and len(tok.lemma_) > 2]
print('Avant   :', tokens_clean[:12])
print('Lemmes  :', tokens_lemme[:12])

### 2.6 Comparaison — Stemming vs Lemmatisation

In [ ]:
phrases_test = [
    "les acteurs jouaient magnifiquement leurs rôles",
    "la connexion entre personnages était parfaitement réussie",
    "les scènes catastrophiques ont énervé profondément",
]

In [ ]:
print(f"{'Texte':<50} {'Stemming':<36} {'Lemmatisation'}")
print('-' * 118)
for ph in phrases_test:
    toks = [t.lower() for t in word_tokenize(ph, language='french')
            if t.isalpha() and t.lower() not in stop_fr and len(t) > 2]
    stems  = [stemmer.stem(t) for t in toks]
    lemmes = [tok.lemma_.lower() for tok in nlp(' '.join(toks))
              if not tok.is_stop and len(tok.lemma_) > 2]
    print(f"{ph:<50} {str(stems):<36} {lemmes}")

### 2.7 Pipeline complète — on retient la lemmatisation

SpaCy est plus lent que le stemming mais produit des tokens linguistiquement corrects.

In [ ]:
def pretraiter(texte):
    texte  = nettoyer(texte)
    tokens = word_tokenize(texte, language='french')
    tokens = [t.lower() for t in tokens if t.isalpha() and len(t) > 2]
    tokens = [t for t in tokens if t not in stop_fr]
    doc    = nlp(' '.join(tokens))
    tokens = [tok.lemma_.lower() for tok in doc if not tok.is_stop and len(tok.lemma_) > 2]
    return ' '.join(tokens)

In [ ]:
print('Prétraitement en cours (~1 min)...')
df_train['texte_clean'] = df_train['review'].apply(pretraiter)
df_test['texte_clean']  = df_test['review'].apply(pretraiter)
print('Terminé.')

In [ ]:
print('BRUT   :', df_train['review'].iloc[0][:200])
print()
print('PROPRE :', df_train['texte_clean'].iloc[0][:200])

---
## Étape 3 — Représentation vectorielle

Les algorithmes ML ne lisent pas des textes — on convertit chaque avis en **vecteur numérique**.

### 3.1 Bag of Words (BoW) — `CountVectorizer`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(df_train['texte_clean'])
X_test_bow  = bow.transform(df_test['texte_clean'])

In [ ]:
print(f'Matrice BoW     : {X_train_bow.shape}')
print(f'Vocabulaire     : {len(bow.get_feature_names_out())} mots')
print(f'Densité         : {X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1]):.4f} (très creuse)')

In [ ]:
vocab_bow = bow.get_feature_names_out()
print('Extrait du vocabulaire (tous les 500 mots) :', vocab_bow[::500])

### 3.2 TF-IDF — `TfidfVectorizer`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(df_train['texte_clean'])
X_test_tfidf  = tfidf.transform(df_test['texte_clean'])
print(f'Matrice TF-IDF : {X_train_tfidf.shape}')

In [ ]:
vocab = tfidf.get_feature_names_out()
pos_mask = (df_train['label'] == 1).values
neg_mask = (df_train['label'] == 0).values
scores_pos = np.asarray(X_train_tfidf[pos_mask].mean(axis=0)).flatten()
scores_neg = np.asarray(X_train_tfidf[neg_mask].mean(axis=0)).flatten()
top_pos = scores_pos.argsort()[-15:][::-1]
top_neg = scores_neg.argsort()[-15:][::-1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(vocab[top_pos][::-1], scores_pos[top_pos][::-1], color='#2ECC71')
axes[0].set_title('Top 15 mots — Avis POSITIFS')
axes[1].barh(vocab[top_neg][::-1], scores_neg[top_neg][::-1], color='#E74C3C')
axes[1].set_title('Top 15 mots — Avis NÉGATIFS')
plt.tight_layout(); plt.show()

---
## Étape 4 — Classification : Naïf de Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_tfidf, df_train['label'])
print('Modèle entraîné.')

In [ ]:
y_pred = nb.predict(X_test_tfidf)
y_test = df_test['label']
print(f'Prédictions sur {len(y_pred)} exemples de test.')

---
## Étape 5 — Évaluation

In [ ]:
from sklearn.metrics import classification_report

print('=== Naïf de Bayes + TF-IDF ===')
print(classification_report(y_test, y_pred, target_names=['Négatif', 'Positif']))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Négatif', 'Positif']).plot(cmap='Blues')
plt.title('Matrice de confusion — Naïf de Bayes')
plt.show()

---
## Étape 6 — Bonus : Comparaison des configurations

On teste 4 combinaisons : **2 représentations** × **2 classifieurs**.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

configurations = [
    ('BoW    + Naïf de Bayes', X_train_bow,   X_test_bow,   MultinomialNB()),
    ('TF-IDF + Naïf de Bayes', X_train_tfidf, X_test_tfidf, MultinomialNB()),
    ('BoW    + SVM linéaire',  X_train_bow,   X_test_bow,   LinearSVC(max_iter=1000)),
    ('TF-IDF + SVM linéaire',  X_train_tfidf, X_test_tfidf, LinearSVC(max_iter=1000)),
]

In [ ]:
resultats = []
for nom, X_tr, X_te, modele in configurations:
    modele.fit(X_tr, df_train['label'])
    pred = modele.predict(X_te)
    resultats.append({'Modèle': nom,
                      'Accuracy': accuracy_score(y_test, pred),
                      'F1 (macro)': f1_score(y_test, pred, average='macro')})
df_res = pd.DataFrame(resultats).set_index('Modèle')
print(df_res.round(3))

In [ ]:
df_res.plot(kind='barh', figsize=(9, 4), color=['#4C72B0', '#DD8452'])
plt.title('Comparaison des configurations')
plt.xlabel('Score')
plt.xlim(0, 1)
plt.tight_layout(); plt.show()

---
## Étape 7 — Tester sur vos propres avis

In [ ]:
svm_final = LinearSVC(max_iter=1000)
svm_final.fit(X_train_tfidf, df_train['label'])
print('Modèle final entraîné.')

In [ ]:
mes_avis = [
    "Un film magnifique, des acteurs exceptionnels, je recommande vivement.",
    "Scénario nul, acteurs mauvais, une heure de perdue.",
    "Pas terrible mais quelques bonnes scènes.",
]
X_mes = tfidf.transform([pretraiter(a) for a in mes_avis])
preds = svm_final.predict(X_mes)
for avis, pred in zip(mes_avis, preds):
    print(f"[{'POSITIF ✓' if pred == 1 else 'NÉGATIF ✗'}]  {avis}")